In [26]:
import numpy as np

x = np.array([1., 2.])
W1 = np.array([[1., 2.], [-1., 1.]])
b1 = np.zeros(2)
w2 = np.array([2., 1.])
b2 = 1.0

def forward(x):
  z1 = x @ W1 + b1
  h = np.maximum(0, z1)
  return z1, h, h @ w2 + b2

z1, h, y_hat = forward(x)
print(z1, h, y_hat)
# [-1.  4.] [0. 4.] 5.0

print("========================")
def backward(x, y):
  z1, h, y_hat = forward(x)
  d = 2 * (y_hat - y)
  dw2 = h * d
  dh = w2 * d
  dz1 = dh * (z1 > 0)
  dW1 = np.outer(x, dz1)
  return dW1, dz1, dw2, d

y = 2.0
dW1, db1, dw2, db2 = backward(x, y)
print(dW1)
print(db1)
print(dw2, db2)
# [[ 0.  6.]
#  [ 0. 12.]]
# [0. 6.]
# [ 0. 24.] 6.0

[-1.  4.] [0. 4.] 5.0
[[ 0.  6.]
 [ 0. 12.]]
[0. 6.]
[ 0. 24.] 6.0


In [12]:
import numpy as np

x = np.array([1., 2.])
W1 = np.array([[1., 2.], [-1., 1.]])
b1 = np.zeros(2)
w2 = np.array([2., 1.])
b2 = 1.0

def forward(x):
  z1 = x @ W1 + b1
  return z1, z1 @ w2 + b2

z1, y_hat = forward(x)
print(z1, y_hat)
# [-1.  4.] 3.0
# Relu-ni olib tashlaganimizda shu chiqdi

[-1.  4.] 3.0


In [13]:
# Loss function
import numpy as np
def mse(y, p):
  return np.mean((p - y) ** 2)

def bce(y, p):
  return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

print(mse(np.array([2.]), np.array([5.])))
for p in (0.9, 0.5, 0.1):
  print(p, round(bce(1, p), 4))
#   9.0
# 0.9 0.1054
# 0.5 0.6931
# 0.1 2.3026

9.0
0.9 0.1054
0.5 0.6931
0.1 2.3026


In [20]:
import numpy as np
def mse(y, p):
  return np.mean((p - y) ** 2)

def bce(y, p):
  return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

print("MSE natija: ", mse(np.array([8., 4., 6.]), np.array([5., 4., 9.])))

for y in (1, 0):
  print(y, "=>", round(bce(y, 0.8), 4))

print(1, "=> ", bce(1, 0.01))

MSE natija:  6.0
1 => 0.2231
0 => 1.6094
1 =>  4.605170185988091


In [28]:
import numpy as np
sigmoid = lambda z: 1 / (1 + np.exp(-z))

class MLP:
  def __init__(self, n_in, n_h, seed=0):
    r = np.random.default_rng(seed)
    self.W1 = r.normal(0, 1, (n_in, n_h))
    self.W2 = r.normal(0, 1, (n_h, 1))
    self.b1, self.b2 = np.zeros(n_h), np.zeros(1)

  def forward(self, X):
    self.h = sigmoid(X @ self.W1 + self.b1)
    self.p = sigmoid(self.h @ self.W2 + self.b2)
    return self.p

  def step(self, X, y, lr):
    dz2 = (self.p - y) / len(X)
    dz1 = dz2 @ self.W2.T * self.h * (1 - self.h)
    self.W2 -= lr * self.h.T @ dz2
    self.b2 -= lr * dz2.sum(0)
    self.W1 -= lr * X.T @ dz1
    self.b1 -= lr * dz1.sum(0)

In [33]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)

net = MLP(2, 4, seed = 1)
for ep in range(5001):
  p = net.forward(X)
  loss = -np.mean(y * np.log(p) + ( 1 - y) * np.log(1 - p))
  if ep in (0, 500, 1000, 2000, 5000):
    print(ep, round(loss, 4))
  net.step(X, y, lr=1.0)


print((net.forward(X) > 0.5).astype(int).ravel())

0 0.7452
500 0.0714
1000 0.014
2000 0.0049
5000 0.0016
[0 1 1 0]


In [34]:
import torch

x = torch.tensor([1., 2.])
W1 = torch.tensor([[1., 2.], [-1., 1.]], requires_grad=True)
b1 = torch.zeros(2, requires_grad=True)
w2 = torch.tensor([2., 1.], requires_grad=True)
b2 = torch.tensor(1., requires_grad=True)

y_hat = torch.relu(x @ W1 + b1) @ w2 + b2
loss = (y_hat - 2) ** 2
loss.backward()

print(W1.grad.numpy())
print(b1.grad.numpy())
print(w2.grad.numpy(), b2.grad.item())
# [[ 0.  6.]
#  [ 0. 12.]]
# [0. 6.]
# [ 0. 24.] 6.0

[[ 0.  6.]
 [ 0. 12.]]
[0. 6.]
[ 0. 24.] 6.0


In [35]:
from sklearn.model_selection import train_test_split
import seaborn as sns

belgilar = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df = sns.load_dataset("penguins").dropna(subset=belgilar)
X = df[belgilar].values
y = df["species"].astype("category").cat.codes.values
print(df.shape, np.bincount(y))

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(len(Xtr), len(Xte))
# (342, 7) [151  68 123]
# 239 103

(342, 7) [151  68 123]
239 103


In [41]:
# pytorch autograd
import torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler

def fit_eval(Xtr, ytr, Xte, yte, scale=True, lr=0.1, hid=8, log=False):
  if scale:
    sc = StandardScaler().fit(Xtr)
    Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

  Xtr = torch.tensor(Xtr).float()
  Xte = torch.tensor(Xte).float()
  ytr = torch.tensor(ytr).long()
  yte = torch.tensor(yte).long()
  torch.manual_seed(42)
  model = nn.Sequential(nn.Linear(4, hid), nn.ReLU(), nn.Linear(hid, 3))
  opt = torch.optim.SGD(model.parameters(), lr=lr)
  for ep in range(201):
    loss = nn.functional.cross_entropy(model(Xtr),  ytr)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if log and ep % 50 == 0:
      print(ep, round(loss.item(), 4))
  acc = (model(Xte).argmax(1) == yte).float().mean()
  return round(acc.item(), 4)

print(fit_eval(Xtr, ytr, Xte, yte, log=True))

0 1.15
50 0.2703
100 0.1403
150 0.0918
200 0.0667
0.9903


In [43]:
# pytorch autograd
import torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler

def fit_eval(Xtr, ytr, Xte, yte, scale=True, lr=0.1, hid=8, log=False):
  # if scale:
    # sc = StandardScaler().fit(Xtr)
    # Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

  Xtr = torch.tensor(Xtr).float()
  Xte = torch.tensor(Xte).float()
  ytr = torch.tensor(ytr).long()
  yte = torch.tensor(yte).long()
  torch.manual_seed(42)
  model = nn.Sequential(nn.Linear(4, hid), nn.ReLU(), nn.Linear(hid, 3))
  opt = torch.optim.SGD(model.parameters(), lr=lr)
  for ep in range(201):
    loss = nn.functional.cross_entropy(model(Xtr),  ytr)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if log and ep % 50 == 0:
      print(ep, round(loss.item(), 4))
  acc = (model(Xte).argmax(1) == yte).float().mean()
  return round(acc.item(), 4)

print(fit_eval(Xtr, ytr, Xte, yte, log=True))

0 113.9657
50 1.0491
100 1.0483
150 1.0482
200 1.0482
0.4369
